# 03 · View Resolution — recover `ViewPosition`, then group frontal / lateral

The multi-view weighted average needs every image assigned to **frontal**
(PA/AP) or **lateral** (LATERAL/LL):

```
p_study = w_f · mean(p over frontal) + w_l · mean(p over lateral),   w_f + w_l = 1
```

`ViewPosition` is missing or non-standard on a slice of MIMIC-CXR. Those images
are usually still frontal or lateral — the field simply was not filled in — and
`mimic_cxr_jpg.metadata` carries four other DICOM fields that may say what the
view actually was.

**This notebook measures before it assigns.** Recovering a view *wrongly* is worse
than dropping the image: the view label is the variable the 7:3 weighting is built
on, so a bad label corrupts the fusion itself rather than merely losing data. C2 is
a hard gate — assignment only happens if the mapping proves deterministic on the
labelled majority.

**It also feeds back into the cohort.** `01`'s attrition dropped **7,292 studies**
on `>=1 usable ViewPosition`. Any study whose view is recoverable here should come
back — so `01` must be re-run afterwards if C5 recovers a meaningful share.


## C0 · Config

In [ ]:
import os, re, numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
CXR     = 'physionet-data.mimic_cxr_jpg'

try:
    from google.colab import auth, drive
    auth.authenticate_user(); drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')

def run_q(sql, label=''):
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True,
                                                              use_query_cache=False))
    print(f'  [{label}] dry-run OK · will scan {dry.total_bytes_processed/1e9:.3f} GB')
    return client.query(sql).to_dataframe()

OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
os.makedirs(OUT_DIR, exist_ok=True)

# purity a recovered mapping must reach before we trust it (C2 / Route 1 / Route 3)
PURITY_MIN = 0.98
# a rule ARM must also be validated on this many labelled images. Pooled accuracy
# alone lets an unsupported arm ride on the majority class -- see C5-bis.
MIN_ARM_SUPPORT = 100
print('OUT_DIR', OUT_DIR)

## C1 · What is actually in `ViewPosition`?

Everything at image level (`dicom_id`), because view is a property of an image, not
of a study.

In [ ]:
vp = run_q(f"""
SELECT COALESCE(NULLIF(TRIM(ViewPosition), ''), '<<NULL/BLANK>>') AS view_position,
       COUNT(*)                   AS n_images,
       COUNT(DISTINCT study_id)   AS n_studies,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 5) AS frac_images
FROM `{CXR}.metadata`
GROUP BY 1 ORDER BY n_images DESC
""", 'viewpos')
print(vp.to_string(index=False))
print(f'\ntotal images: {vp.n_images.sum():,}')
missing = vp.loc[vp.view_position == '<<NULL/BLANK>>', 'n_images'].sum()
print(f'missing ViewPosition: {missing:,} images ({missing/vp.n_images.sum():.3%})')

## C2 · GATE — is `ViewPosition` recoverable from `ViewCodeSequence`?

The whole recovery rests on one assumption: that `ViewCodeSequence_CodeMeaning`
determines `ViewPosition`. Test it **on the rows where both are present** — if each
code maps to one dominant position with ≥`PURITY_MIN` purity, the mapping inverts
safely. If a code splits across positions, it is ambiguous and must **not** be used
to label anything.

Purity is computed per code, weighted by image count. We check
`PatientOrientationCodeSequence_CodeMeaning` the same way, as a second witness.

In [ ]:
def purity_table(field, label):
    t = run_q(f"""
    SELECT COALESCE(NULLIF(TRIM({field}), ''), '<<null>>') AS code,
           TRIM(ViewPosition)                              AS view_position,
           COUNT(*)                                        AS n
    FROM `{CXR}.metadata`
    WHERE NULLIF(TRIM(ViewPosition), '') IS NOT NULL
    GROUP BY 1, 2
    """, label)
    tot  = t.groupby('code')['n'].sum().rename('n_total')
    top  = (t.sort_values('n', ascending=False).groupby('code')
              .first()[['view_position', 'n']]
              .rename(columns={'view_position': 'majority_view', 'n': 'n_majority'}))
    out = top.join(tot)
    out['purity']   = (out.n_majority / out.n_total).round(4)
    out['n_other']  = out.n_total - out.n_majority
    out['verdict']  = np.where(out.purity >= PURITY_MIN, 'usable', 'AMBIGUOUS')
    return t, out.sort_values('n_total', ascending=False)

raw_vc, pur_vc = purity_table('ViewCodeSequence_CodeMeaning', 'purity-viewcode')
print('=== ViewCodeSequence_CodeMeaning -> ViewPosition ===')
print(pur_vc.to_string())

w = (pur_vc.n_majority.sum() / pur_vc.n_total.sum())
print(f'\noverall weighted purity: {w:.4f}   (gate: >= {PURITY_MIN})')
bad = pur_vc[pur_vc.verdict == 'AMBIGUOUS']
if len(bad):
    print(f'!! {len(bad)} ambiguous code(s) -- these will NOT be used for recovery:')
    print(bad.to_string())

In [ ]:
raw_or, pur_or = purity_table('PatientOrientationCodeSequence_CodeMeaning', 'purity-orient')
print('=== PatientOrientationCodeSequence_CodeMeaning -> ViewPosition (second witness) ===')
print(pur_or.to_string())
print(f'\noverall weighted purity: {pur_or.n_majority.sum()/pur_or.n_total.sum():.4f}')

## C3 · What do the *missing-view* images actually carry?

Purity on labelled rows means nothing if the unlabelled rows have no code either.
This is the coverage side of the question.

In [ ]:
cov = run_q(f"""
SELECT
  COUNT(*)                                                                       AS n_missing_view,
  COUNTIF(NULLIF(TRIM(ViewCodeSequence_CodeMeaning), '') IS NOT NULL)            AS has_view_code,
  COUNTIF(NULLIF(TRIM(PatientOrientationCodeSequence_CodeMeaning), '') IS NOT NULL) AS has_orientation,
  COUNTIF(NULLIF(TRIM(PerformedProcedureStepDescription), '') IS NOT NULL)       AS has_proc_desc,
  COUNTIF(NULLIF(TRIM(ViewCodeSequence_CodeMeaning), '') IS NOT NULL
       OR NULLIF(TRIM(PatientOrientationCodeSequence_CodeMeaning), '') IS NOT NULL) AS recoverable_any
FROM `{CXR}.metadata`
WHERE NULLIF(TRIM(ViewPosition), '') IS NULL
""", 'coverage').iloc[0]
print(cov.to_string())
n_miss = int(cov.n_missing_view)
if n_miss:
    print(f'\nrecoverable from any witness: {int(cov.recoverable_any):,} / {n_miss:,} '
          f'= {cov.recoverable_any/n_miss:.2%}')

detail = run_q(f"""
SELECT
  COALESCE(NULLIF(TRIM(ViewCodeSequence_CodeMeaning), ''), '<<null>>')               AS view_code,
  COALESCE(NULLIF(TRIM(PatientOrientationCodeSequence_CodeMeaning), ''), '<<null>>') AS orientation,
  COALESCE(NULLIF(TRIM(PerformedProcedureStepDescription), ''), '<<null>>')          AS proc_desc,
  COUNT(*) AS n_images
FROM `{CXR}.metadata`
WHERE NULLIF(TRIM(ViewPosition), '') IS NULL
GROUP BY 1,2,3 ORDER BY n_images DESC LIMIT 25
""", 'missing-detail')
print('\nwhat the missing-view images say instead:')
print(detail.to_string(index=False))

## C4 · Recover — only through codes that passed the gate

`ViewCodeSequence` first (higher purity expected), `PatientOrientation` as fallback.
Anything still unresolved stays `<<unresolved>>` and is **not guessed**.

In [ ]:
map_vc = {c: r.majority_view for c, r in pur_vc.iterrows()
          if r.verdict == 'usable' and c != '<<null>>'}
map_or = {c: r.majority_view for c, r in pur_or.iterrows()
          if r.verdict == 'usable' and c != '<<null>>'}
print(f'usable ViewCode mappings   : {len(map_vc)}')
print(f'usable Orientation mappings: {len(map_or)}')

img = run_q(f"""
SELECT dicom_id, study_id, subject_id,
       NULLIF(TRIM(ViewPosition), '')                               AS view_position,
       NULLIF(TRIM(ViewCodeSequence_CodeMeaning), '')               AS view_code,
       NULLIF(TRIM(PatientOrientationCodeSequence_CodeMeaning), '') AS orientation,
       NULLIF(TRIM(PerformedProcedureStepDescription), '')          AS proc_desc,
       `Rows`, `Columns`          -- ROWS is a reserved keyword in BigQuery -> must be quoted
FROM `{CXR}.metadata`
""", 'images')
print(f'\nimages: {len(img):,}')

src = np.where(img.view_position.notna(), 'original', '')
vpos = img.view_position.copy()

m = vpos.isna() & img.view_code.map(lambda x: x in map_vc)
vpos = vpos.mask(m, img.view_code.map(map_vc)); src = np.where(m, 'ViewCodeSequence', src)

m2 = vpos.isna() & img.orientation.map(lambda x: x in map_or)
vpos = vpos.mask(m2, img.orientation.map(map_or)); src = np.where(m2, 'PatientOrientation', src)

img['view_position_resolved'] = vpos
img['view_source'] = np.where(vpos.isna(), 'unresolved', src)
print('\nresolution source:')
print(img.view_source.value_counts().to_frame('n_images')
        .assign(frac=lambda d: (d.n_images/len(img)).round(4)).to_string())

## C5 · Group into `frontal` / `lateral` / `other`

`frontal` = PA, AP and positional variants (AP AXIAL / LLD / RLD). `lateral` =
LATERAL, LL, XTABLE LATERAL. Oblique and swimmer views stay `other` — they are not
what `w_f`/`w_l` were defined over, and forcing them in would silently redefine the
weighting. Measured: all of them together are **21 images** out of 377,110, so this
costs nothing.

**Only the group matters.** The 7:3 average is frontal-vs-lateral, so we never have
to decide AP-vs-PA — which is exactly what makes the procedure-description route
below tractable.

In [ ]:
FRONTAL_RE = re.compile(r'^(PA|AP)\b', re.I)
LATERAL_RE = re.compile(r'(LATERAL|^LL$)', re.I)

def group(v):
    if not isinstance(v, str): return 'other'
    v = v.strip().upper()
    if LATERAL_RE.search(v): return 'lateral'
    if FRONTAL_RE.search(v): return 'frontal'
    return 'other'

img['view_group'] = img.view_position_resolved.map(group)
print('after witness-code recovery only (C4):')
print(img.view_group.value_counts().to_frame('n_images')
        .assign(frac=lambda d: (d.n_images/len(img)).round(4)).to_string())

## C5-bis · ROUTE 1 — procedure description, **validated before use**

`ViewCodeSequence` recovered nothing (0% coverage — the missing-view images have no
code either). But `PerformedProcedureStepDescription` is present on **92.8%** of
them. It describes the *order*, not the image, so it can only resolve an image when
the order names exactly one view family:

| the order says | image group |
|---|---|
| a frontal view and **no** lateral (`CHEST (PORTABLE AP)`, `CHEST (SINGLE VIEW)`) | `frontal` |
| a lateral and **no** frontal (`CHEST (LAT DECUB ONLY) PORT`) | `lateral` |
| **both** (`CHEST (PA AND LAT)`, 78% of the missing) | undecidable per-image → Route 2 |

**Validated first, like C2:** run the rule on images that *do* have a real
`ViewPosition` and check it reproduces their group. A rule that cannot recover known
labels does not get to invent unknown ones.

In [ ]:
def proc_to_group(p):
    # Order-level text -> image group. Decides only when the order names ONE family.
    if not isinstance(p, str): return None
    u = p.upper()
    if not re.search(r'CHEST|RIB|BABYGRAM', u): return None       # not a chest order
    has_lat = bool(re.search(r'\bLAT\b|LATERAL|DECUB', u))
    has_fr  = bool(re.search(r'\bAP\b|\bPA\b|SINGLE VIEW', u))
    if has_fr and not has_lat: return 'frontal'
    if has_lat and not has_fr: return 'lateral'
    return None                                                    # both, or neither

lab = img[img.view_position.notna() & img.view_group.isin(['frontal','lateral'])].copy()
lab['proc_pred'] = lab.proc_desc.map(proc_to_group)
dec = lab[lab.proc_pred.notna()]
acc = (dec.proc_pred == dec.view_group).mean() if len(dec) else 0.0
print('Route 1 validation on labelled images:')
print(f'   decided by proc_desc : {len(dec):,} / {len(lab):,}  ({len(dec)/max(len(lab),1):.2%})')
print(f'   POOLED agreement     : {acc:.4f}')
if len(dec):
    print('\n   confusion (rows = proc says, cols = truth):')
    print(pd.crosstab(dec.proc_pred, dec.view_group).to_string())

# --- PER-ARM gate. A pooled accuracy is dominated by the majority class and will
#     wave through an arm that has no evidence at all: on MIMIC-CXR the `lateral`
#     arm fires on ONE labelled image and gets it wrong, yet pooled acc = 0.9948.
#     Each arm must clear the gate on its OWN support before it may label anything.
per_arm = {}
for g in ('frontal', 'lateral'):
    sub = dec[dec.proc_pred == g]
    n   = len(sub)
    a   = float((sub.view_group == g).mean()) if n else 0.0
    ok  = bool(n >= MIN_ARM_SUPPORT and a >= PURITY_MIN)
    per_arm[g] = dict(n=n, acc=a, enabled=ok)
print(f'\n   per-arm gate (need n >= {MIN_ARM_SUPPORT} AND acc >= {PURITY_MIN}):')
for g, v in per_arm.items():
    why = ('ok' if v['enabled'] else
           ('too few labelled examples to trust' if v['n'] < MIN_ARM_SUPPORT else 'accuracy below gate'))
    print(f"      {g:<8} n={v['n']:>7,}  acc={v['acc']:.4f}  -> "
          f"{'ENABLED ' if v['enabled'] else 'REJECTED'}  ({why})")

ENABLED_ARMS = [g for g, v in per_arm.items() if v['enabled']]
ROUTE1_OK = len(ENABLED_ARMS) > 0
print(f'\n   -> Route 1 arms in use: {ENABLED_ARMS if ENABLED_ARMS else "NONE"}')
if ROUTE1_OK:
    pred = img.proc_desc.map(proc_to_group)
    m = img.view_position_resolved.isna() & pred.isin(ENABLED_ARMS)   # rejected arms abstain
    img.loc[m, 'view_group']  = pred[m]
    img.loc[m, 'view_source'] = 'ProcedureDesc'
    print(f'   recovered {int(m.sum()):,} images from the procedure description')
    print(f'   (images whose proc points at a REJECTED arm stay unresolved)')

## C5-ter · ROUTE 2 — elimination inside a study

For `CHEST (PA AND LAT)` the order names both families, so it cannot label an image
by itself. But it constrains the *study*: if the order says PA **and** LAT, the
study's resolved images already cover `frontal`, and exactly one image is left
unresolved — then that image **is** the lateral. And symmetrically.

This is deduction, not inference: no threshold, no model. It fires only when the
constraint leaves one possibility and abstains otherwise — e.g. a two-view order
with *both* images unlabelled, where the set is known to be {frontal, lateral} but
the assignment is not.

In [ ]:
def proc_families(p):
    if not isinstance(p, str): return (False, False)
    u = p.upper()
    if not re.search(r'CHEST|RIB|BABYGRAM', u): return (False, False)
    return (bool(re.search(r'\bAP\b|\bPA\b|SINGLE VIEW', u)),
            bool(re.search(r'\bLAT\b|LATERAL|DECUB', u)))

img['_f_ord'], img['_l_ord'] = zip(*img.proc_desc.map(proc_families))
st = img.groupby('study_id').agg(
        f_ord=('_f_ord','max'), l_ord=('_l_ord','max'),
        n_unres=('view_group', lambda s: (s == 'other').sum()),
        has_f=('view_group', lambda s: (s == 'frontal').any()),
        has_l=('view_group', lambda s: (s == 'lateral').any()))

both = st.f_ord & st.l_ord
elim_l = set(st[both & st.has_f & ~st.has_l & (st.n_unres == 1)].index)
elim_f = set(st[both & st.has_l & ~st.has_f & (st.n_unres == 1)].index)
print(f'studies where elimination yields LATERAL: {len(elim_l):,}')
print(f'studies where elimination yields FRONTAL: {len(elim_f):,}')

m_l = (img.view_group == 'other') & img.study_id.isin(elim_l)
m_f = (img.view_group == 'other') & img.study_id.isin(elim_f)
img.loc[m_l, 'view_group'] = 'lateral'; img.loc[m_l, 'view_source'] = 'Elimination'
img.loc[m_f, 'view_group'] = 'frontal'; img.loc[m_f, 'view_source'] = 'Elimination'
print(f'recovered by elimination: {int(m_l.sum() + m_f.sum()):,} images')
print(f'\nstill unresolved: {int((img.view_group == "other").sum()):,} images')
print('   (mostly two-view orders with BOTH images unlabelled: the set is known,')
print('    the assignment is not -> Route 3 decides those, if it earns it.)')

## C5-quater · ROUTE 3 — aspect ratio, only if it earns it

What remains are orders naming PA **and** LAT with **both** images unlabelled;
elimination cannot split those. A lateral chest film has a different height/width
ratio from a frontal one, so `Rows/Columns` *might* separate them.

*Might* — so measure it. Fit the best single threshold on labelled images, report its
accuracy, and apply it **only** if it clears `PURITY_MIN`. Otherwise these images
keep `other` and simply sit out the weighted average: cheaper than mislabelling the
one variable the whole 7:3 scheme rests on.

In [ ]:
lab2 = img[img.view_position.notna() & img.view_group.isin(['frontal','lateral'])].copy()
lab2['ar'] = lab2.Rows / lab2.Columns
lab2 = lab2[np.isfinite(lab2.ar)]
print('aspect ratio (Rows/Columns) by TRUE group:')
print(lab2.groupby('view_group').ar.describe(percentiles=[.1,.5,.9])
        [['count','mean','10%','50%','90%']].round(3).to_string())

best_t, best_acc, best_dir = None, 0.0, None
for t in np.arange(0.60, 2.00, 0.01):
    for d, pred in (('lat_high', np.where(lab2.ar >= t, 'lateral', 'frontal')),
                    ('lat_low',  np.where(lab2.ar <  t, 'lateral', 'frontal'))):
        a = (pred == lab2.view_group).mean()
        if a > best_acc: best_t, best_acc, best_dir = float(t), float(a), d
sign = '>=' if best_dir == 'lat_high' else '<'
print(f'\nbest single threshold: ar {sign} {best_t:.2f} -> lateral')
print(f'accuracy {best_acc:.4f}   (gate: >= {PURITY_MIN})')

ROUTE3_OK = bool(best_acc >= PURITY_MIN)
print(f'   -> Route 3 {"ENABLED" if ROUTE3_OK else "REJECTED -- aspect ratio does not separate"}')
if ROUTE3_OK:
    m = img.view_group == 'other'
    ar = (img.loc[m, 'Rows'] / img.loc[m, 'Columns']).to_numpy()
    pred = (np.where(ar >= best_t, 'lateral', 'frontal') if best_dir == 'lat_high'
            else np.where(ar < best_t, 'lateral', 'frontal'))
    img.loc[m, 'view_group']  = pred
    img.loc[m, 'view_source'] = 'AspectRatio'
    print(f'   recovered {int(m.sum()):,} images by aspect ratio')
else:
    print('   these keep view_group="other" and are excluded from the 7:3 average.')

print('\n=== FINAL view_group ===')
print(img.view_group.value_counts().to_frame('n_images')
        .assign(frac=lambda d: (d.n_images/len(img)).round(4)).to_string())
print('\nrecovery source:')
print(img.view_source.value_counts().to_frame('n_images')
        .assign(frac=lambda d: (d.n_images/len(img)).round(4)).to_string())

## C6 · Impact on the cohort — do the 7,292 dropped studies come back?

`01` required `>=1 usable ViewPosition`, using the raw field. With recovery, a study
is usable if it has `>=1` image in **frontal or lateral**. This cell quantifies what
re-running `01` would change.

In [ ]:
per_study = (img.assign(f=img.view_group.eq('frontal'), l=img.view_group.eq('lateral'))
                .groupby('study_id')
                .agg(n_images=('dicom_id','size'), n_frontal=('f','sum'), n_lateral=('l','sum'),
                     n_raw_view=('view_position', lambda s: s.notna().sum())))
per_study['usable_old'] = per_study.n_raw_view >= 1
per_study['usable_new'] = (per_study.n_frontal + per_study.n_lateral) >= 1

print(pd.crosstab(per_study.usable_old, per_study.usable_new,
                  rownames=['usable_old (raw field)'], colnames=['usable_new (recovered)'],
                  margins=True).to_string())
gained = int(((~per_study.usable_old) & per_study.usable_new).sum())
lost   = int((per_study.usable_old & (~per_study.usable_new)).sum())
print(f'\nstudies RECOVERED into the cohort : {gained:,}')
print(f'studies that would now DROP OUT   : {lost:,}   '
      '(had a raw view, but it is oblique/swimmer -> neither frontal nor lateral)')
if gained:
    print(f'\n-> re-run 01 (and therefore 02) so the cohort includes these {gained:,} studies.')
else:
    print('\n-> no cohort change; 01 does not need re-running for view reasons.')

print('\nper-study view composition (frontal/lateral availability drives the 7:3 weights):')
comp = pd.Series(np.select(
    [(per_study.n_frontal>0) & (per_study.n_lateral>0),
     (per_study.n_frontal>0) & (per_study.n_lateral==0),
     (per_study.n_frontal==0) & (per_study.n_lateral>0)],
    ['both frontal+lateral','frontal only','lateral only'], default='neither'))
print(comp.value_counts().to_frame('n_studies')
        .assign(frac=lambda d: (d.n_studies/len(per_study)).round(4)).to_string())
print('\n!! "frontal only" is the majority -> w_f/w_l MUST be renormalised over the')
print('   view groups actually present, or those studies get a systematically')
print('   shrunken embedding (0.7x) / deflated probability. See 04 and 05.')

## C7 · Emit `view_map.parquet`

In [ ]:
out = img[['dicom_id','study_id','subject_id','view_position','view_position_resolved',
           'view_source','view_group']]
out.to_parquet(os.path.join(OUT_DIR,'view_map.parquet'), index=False)
per_study.reset_index().to_parquet(os.path.join(OUT_DIR,'view_per_study.parquet'), index=False)

for f in ['view_map.parquet','view_per_study.parquet']:
    b = os.path.getsize(os.path.join(OUT_DIR,f))
    print(f'  {b:,} B  {f}')
print(f'\nimages: {len(out):,} | studies: {len(per_study):,}')
print('\nrecovery by route (this line used to count only the witness fields and so')
print('under-reported every image Routes 1-3 actually recovered):')
RECOVERY_SRC = ['ViewCodeSequence','PatientOrientation','ProcedureDesc','Elimination','AspectRatio']
for srcname in RECOVERY_SRC:
    n = int((out.view_source == srcname).sum())
    if n: print(f'   {srcname:<20} {n:>7,}')
tot = int(out.view_source.isin(RECOVERY_SRC).sum())
print(f'   {"TOTAL recovered":<20} {tot:>7,}  ({tot/len(out):.2%} of images)')
print(f'   {"still unresolved":<20} {int((out.view_source=="unresolved").sum()):>7,}')
print('\nNEXT: 04_raddino_lpft.ipynb  (ML-Decoder head + DB loss, eval with w_f/w_l)')